# TinyML Keyword Spotting: FP32 vs INT8 CNN
**Goal:** Train a small CNN on five keyword classes, convert it to a fully INT8 TensorFlow Lite model, and compare accuracy, model size, latency, and predictions.

**Scope:** Software simulation in Google Colab. This does not claim physical ESP32 deployment or ESP32 timing/power measurements.

Dataset: TensorFlow's `mini_speech_commands` (five classes: no, yes, down, go, up).

## 1. Install/import libraries
Run cells from top to bottom. If Colab asks to restart the runtime after installation, do so and continue from the imports cell.

In [ ]:
!pip -q install scikit-learn seaborn pandas matplotlib

In [ ]:
import os, pathlib, time, zipfile, urllib.request, json, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 2. Download the small dataset
The archive is downloaded from the TensorFlow-hosted tutorial dataset. It contains short WAV recordings across five keywords.

In [ ]:
DATA_URL = "https://storage.googleapis.com/download.tensorflow.org/data/mini_speech_commands.zip"
WORKDIR = pathlib.Path("/content/tinyml_keyword_spotting")
WORKDIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = WORKDIR / "mini_speech_commands.zip"
DATA_DIR = WORKDIR / "mini_speech_commands"

if not DATA_DIR.exists():
    if not ZIP_PATH.exists():
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(WORKDIR)

print("Dataset directory:", DATA_DIR)
print("Keyword folders:", sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()]))

## 3. Build reproducible train/validation/test splits
We use only the five selected keyword folders. Audio is converted to a fixed 16 kHz, one-second waveform. The model will consume a spectrogram.

In [ ]:
LABELS = ["no", "yes", "down", "go", "up"]
SAMPLE_RATE = 16000
AUDIO_LENGTH = SAMPLE_RATE  # one second
AUTOTUNE = tf.data.AUTOTUNE

paths, labels = [], []
for label_id, label in enumerate(LABELS):
    files = sorted((DATA_DIR / label).glob("*.wav"))
    for f in files:
        paths.append(str(f))
        labels.append(label_id)

paths = np.array(paths)
labels = np.array(labels, dtype=np.int64)
print("Total selected clips:", len(paths))
print(pd.Series([LABELS[i] for i in labels]).value_counts())

# Split file paths before loading to reduce RAM use.
train_paths, temp_paths, y_train, y_temp = train_test_split(
    paths, labels, test_size=0.30, random_state=SEED, stratify=labels
)
val_paths, test_paths, y_val, y_test = train_test_split(
    temp_paths, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)
print("Train / validation / test:", len(train_paths), len(val_paths), len(test_paths))

## 4. Audio preprocessing and spectrogram creation
All clips are made exactly one second long. Spectrograms are generated with an STFT and log-scaled. We use the same preprocessing for training and both model comparisons.

In [ ]:
FRAME_LENGTH = 256
FRAME_STEP = 128
FFT_LENGTH = 256

def load_audio(path, label):
    audio_bytes = tf.io.read_file(path)
    audio, sr = tf.audio.decode_wav(audio_bytes, desired_channels=1)
    audio = tf.squeeze(audio, axis=-1)
    audio = tf.cast(audio, tf.float32)

    # The mini_speech_commands WAV files are 16 kHz. Fail clearly if that changes.
    tf.debugging.assert_equal(
        tf.cast(sr, tf.int32), SAMPLE_RATE,
        message="Expected 16 kHz audio in mini_speech_commands."
    )
    audio = audio[:AUDIO_LENGTH]
    padding = tf.maximum(0, AUDIO_LENGTH - tf.shape(audio)[0])
    audio = tf.pad(audio, [[0, padding]])
    audio.set_shape([AUDIO_LENGTH])

    stft = tf.signal.stft(audio, frame_length=FRAME_LENGTH,
                          frame_step=FRAME_STEP, fft_length=FFT_LENGTH)
    spec = tf.abs(stft)
    spec = tf.math.log(spec + 1e-6)
    spec = spec[..., tf.newaxis]
    return spec, tf.cast(label, tf.int32)

def make_ds(p, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((p, y))
    if training:
        ds = ds.shuffle(len(p), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_audio, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(32).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(train_paths, y_train, training=True)
val_ds = make_ds(val_paths, y_val)
test_ds = make_ds(test_paths, y_test)

for xb, yb in train_ds.take(1):
    print("Spectrogram batch shape:", xb.shape, "labels shape:", yb.shape)
    plt.figure(figsize=(8, 3))
    plt.imshow(xb[0, :, :, 0].numpy().T, aspect="auto", origin="lower")
    plt.title("Example log-magnitude spectrogram")
    plt.xlabel("Time frame")
    plt.ylabel("Frequency bin")
    plt.colorbar()
    plt.show()

## 5. Define and train a small CNN
This is a compact baseline model, not a claim of state-of-the-art accuracy. The held-out test set is reserved for evaluation.

In [ ]:
input_shape = next(iter(train_ds))[0].shape[1:]

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=input_shape, name="spectrogram"),
    tf.keras.layers.Conv2D(8, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(len(LABELS), activation="softmax")
], name="tiny_keyword_cnn")

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

EPOCHS = 12  # If time is tight, try 6-8 first; inspect validation accuracy.
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)

os.makedirs("artifacts", exist_ok=True)
model.save("artifacts/keyword_cnn_fp32.keras")

## 6. Plot training curves and evaluate the FP32 baseline

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history.history["accuracy"], label="train")
ax[0].plot(history.history["val_accuracy"], label="validation")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(history.history["loss"], label="train")
ax[1].plot(history.history["val_loss"], label="validation")
ax[1].set_title("Loss"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

fp32_loss, fp32_keras_acc = model.evaluate(test_ds, verbose=0)
print(f"FP32 Keras test accuracy: {fp32_keras_acc:.4f}")

## 7. Export a float32 TensorFlow Lite model
We use a concrete inference function so the saved TFLite model has a known input shape.

In [ ]:
@tf.function(input_signature=[tf.TensorSpec(shape=[1, *input_shape], dtype=tf.float32)])
def serving_fn(x):
    return {"probabilities": model(x, training=False)}

concrete_fn = serving_fn.get_concrete_function()
converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_fn], model)
fp32_tflite = converter.convert()
open("artifacts/keyword_cnn_fp32.tflite", "wb").write(fp32_tflite)
print("FP32 TFLite bytes:", len(fp32_tflite))

## 8. Convert to fully INT8 TensorFlow Lite
Representative examples calibrate activation ranges. `TFLITE_BUILTINS_INT8` and INT8 input/output enforce integer-only built-in operations at the model boundary and for supported ops. Conversion will fail rather than silently retain unsupported float ops.

In [ ]:
def representative_dataset():
    # Use a subset of training spectrograms for calibration.
    for specs, _ in train_ds.take(100):
        for i in range(specs.shape[0]):
            yield [tf.cast(specs[i:i+1], tf.float32)]

int8_converter = tf.lite.TFLiteConverter.from_concrete_functions(
    [concrete_fn], model
)
int8_converter.optimizations = [tf.lite.Optimize.DEFAULT]
int8_converter.representative_dataset = representative_dataset
int8_converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]
int8_converter.inference_input_type = tf.int8
int8_converter.inference_output_type = tf.int8

int8_tflite = int8_converter.convert()
open("artifacts/keyword_cnn_int8.tflite", "wb").write(int8_tflite)
print("INT8 TFLite bytes:", len(int8_tflite))

## 9. TFLite inference helpers
The FP32 model accepts float spectrograms. The INT8 model requires explicit quantization using the input tensor's scale and zero-point, and output dequantization for interpretable scores.

In [ ]:
def make_interpreter(model_content):
    interpreter = tf.lite.Interpreter(model_content=model_content)
    interpreter.allocate_tensors()
    return interpreter

fp32_interpreter = make_interpreter(fp32_tflite)
int8_interpreter = make_interpreter(int8_tflite)

def get_quant_params(detail):
    scale, zero_point = detail["quantization"]
    if scale == 0:
        raise ValueError("Invalid quantization scale (zero).")
    return float(scale), int(zero_point)

def predict_one(interpreter, spec, integer_model=False):
    # spec shape is [frames, frequency_bins, 1]
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    x = spec[np.newaxis, ...].astype(np.float32)

    if integer_model:
        scale, zero = get_quant_params(inp)
        dtype = inp["dtype"]
        limits = np.iinfo(dtype)
        x = np.round(x / scale + zero)
        x = np.clip(x, limits.min, limits.max).astype(dtype)
    else:
        x = x.astype(inp["dtype"])

    interpreter.set_tensor(inp["index"], x)
    interpreter.invoke()
    pred = interpreter.get_tensor(out["index"])

    if integer_model and out["dtype"] in (np.int8, np.uint8):
        out_scale, out_zero = get_quant_params(out)
        pred = (pred.astype(np.float32) - out_zero) * out_scale
    else:
        pred = pred.astype(np.float32)
    return pred[0]

print("FP32 input:", fp32_interpreter.get_input_details()[0]["dtype"],
      fp32_interpreter.get_input_details()[0]["quantization"])
print("INT8 input:", int8_interpreter.get_input_details()[0]["dtype"],
      int8_interpreter.get_input_details()[0]["quantization"])
print("INT8 output:", int8_interpreter.get_output_details()[0]["dtype"],
      int8_interpreter.get_output_details()[0]["quantization"])

## 10. Compare FP32 TFLite and INT8 TFLite on exactly the same test samples
This evaluates the exported models, not just the original Keras model.

In [ ]:
# Materialize the test spectrograms and labels once for paired comparisons.
X_test_list, y_test_list = [], []
for xb, yb in test_ds:
    X_test_list.extend(xb.numpy())
    y_test_list.extend(yb.numpy())
X_test = np.asarray(X_test_list, dtype=np.float32)
y_test_np = np.asarray(y_test_list, dtype=np.int64)

def batch_predict(interpreter, X, integer_model=False):
    return np.stack([predict_one(interpreter, x, integer_model) for x in X])

fp32_probs = batch_predict(fp32_interpreter, X_test, integer_model=False)
int8_probs = batch_predict(int8_interpreter, X_test, integer_model=True)
fp32_pred = np.argmax(fp32_probs, axis=1)
int8_pred = np.argmax(int8_probs, axis=1)

fp32_acc = float(np.mean(fp32_pred == y_test_np))
int8_acc = float(np.mean(int8_pred == y_test_np))
agreement = float(np.mean(fp32_pred == int8_pred))

print(f"FP32 TFLite test accuracy: {fp32_acc:.4f}")
print(f"INT8 TFLite test accuracy: {int8_acc:.4f}")
print(f"Prediction agreement:      {agreement:.4f}")
print(f"Accuracy change (INT8-FP32): {(int8_acc-fp32_acc)*100:.2f} percentage points")
print("\nINT8 classification report:")
print(classification_report(y_test_np, int8_pred, target_names=LABELS, zero_division=0))

## 11. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, pred, title in [
    (axes[0], fp32_pred, "FP32 TFLite"),
    (axes[1], int8_pred, "INT8 TFLite")
]:
    cm = confusion_matrix(y_test_np, pred, labels=np.arange(len(LABELS)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=LABELS, yticklabels=LABELS, ax=ax)
    ax.set_title(title); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig("artifacts/confusion_matrices.png", dpi=160, bbox_inches="tight")
plt.show()

## 12. Measure inference latency
This is **Colab host CPU latency**, not ESP32 latency. We warm up each interpreter and time individual single-example inferences. Results vary with runtime, CPU load, and TensorFlow version.

In [ ]:
def benchmark(interpreter, X, integer_model=False, warmup=10, repeats=3):
    n = min(len(X), 200)
    samples = X[:n]
    for x in samples[:min(warmup, n)]:
        predict_one(interpreter, x, integer_model)
    timings_ms = []
    for _ in range(repeats):
        start = time.perf_counter()
        for x in samples:
            predict_one(interpreter, x, integer_model)
        elapsed = time.perf_counter() - start
        timings_ms.append(elapsed / n * 1000)
    return float(np.median(timings_ms)), float(np.percentile(timings_ms, 90))

fp32_ms, fp32_p90 = benchmark(fp32_interpreter, X_test, False)
int8_ms, int8_p90 = benchmark(int8_interpreter, X_test, True)
print(f"FP32 TFLite latency: median {fp32_ms:.3f} ms/sample; p90 {fp32_p90:.3f} ms")
print(f"INT8 TFLite latency: median {int8_ms:.3f} ms/sample; p90 {int8_p90:.3f} ms")

## 13. Model size and summary table

In [ ]:
fp32_size = os.path.getsize("artifacts/keyword_cnn_fp32.tflite")
int8_size = os.path.getsize("artifacts/keyword_cnn_int8.tflite")
size_reduction = (1 - int8_size / fp32_size) * 100

results = pd.DataFrame([
    {"Model": "FP32 TFLite", "Test accuracy": fp32_acc,
     "Model size (KB)": fp32_size / 1024, "Median latency (ms)": fp32_ms,
     "P90 latency (ms)": fp32_p90},
    {"Model": "INT8 TFLite", "Test accuracy": int8_acc,
     "Model size (KB)": int8_size / 1024, "Median latency (ms)": int8_ms,
     "P90 latency (ms)": int8_p90},
])
display(results.round(4))
print(f"Model-size reduction: {size_reduction:.2f}%")

results.to_csv("artifacts/fp32_vs_int8_results.csv", index=False)
with open("artifacts/metrics.json", "w") as f:
    json.dump({
        "labels": LABELS,
        "fp32_tflite_accuracy": fp32_acc,
        "int8_tflite_accuracy": int8_acc,
        "prediction_agreement": agreement,
        "fp32_size_bytes": fp32_size,
        "int8_size_bytes": int8_size,
        "size_reduction_percent": size_reduction,
        "fp32_median_latency_ms_colab_cpu": fp32_ms,
        "int8_median_latency_ms_colab_cpu": int8_ms,
        "latency_note": "Colab host CPU measurements; not ESP32 measurements."
    }, f, indent=2)

## 14. Plot the comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
names = ["FP32", "INT8"]
axes[0].bar(names, [fp32_acc*100, int8_acc*100])
axes[0].set_title("Test accuracy"); axes[0].set_ylabel("%")
axes[1].bar(names, [fp32_size/1024, int8_size/1024])
axes[1].set_title("TFLite model size"); axes[1].set_ylabel("KB")
axes[2].bar(names, [fp32_ms, int8_ms])
axes[2].set_title("Median host latency"); axes[2].set_ylabel("ms/sample")
plt.tight_layout()
plt.savefig("artifacts/fp32_vs_int8_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

## 15. Show a few paired predictions

In [ ]:
for i in range(min(10, len(X_test))):
    true_name = LABELS[y_test_np[i]]
    f_name = LABELS[fp32_pred[i]]
    q_name = LABELS[int8_pred[i]]
    print(f"Sample {i:02d}: true={true_name:>4} | FP32={f_name:>4} | INT8={q_name:>4}")

## 16. Save source model and package artifacts
The ZIP includes trained model files, plots, and metrics. Download it from the Colab Files pane or use the optional download cell below.

In [ ]:
# Include the main artifacts; training source remains in this notebook.
shutil.make_archive("/content/tinyml_keyword_spotting_artifacts", "zip", "artifacts")
print("Created /content/tinyml_keyword_spotting_artifacts.zip")

In [ ]:
from google.colab import files
files.download("/content/tinyml_keyword_spotting_artifacts.zip")

## 17. Interpretation template (fill in using your actual results)
- FP32 TFLite test accuracy: **[your measured result]**
- INT8 TFLite test accuracy: **[your measured result]**
- Model-size reduction: **[your measured result]%**
- Host CPU median latency: FP32 **[result] ms**, INT8 **[result] ms**
- Prediction agreement: **[your measured result]%**

Discuss whether quantization reduced size, whether accuracy changed, and whether latency improved on the Colab CPU. Do not claim lower ESP32 power, ESP32 latency, or physical deployment without measuring it on the actual board.